# Query-qaware: evaluate and benchmark a finetuned checkpoint

Run the cells in order on Kaggle with Internet and at least one GPU enabled.
In the settings cell, fill CHECKPOINT_HF_REPO_ID + CHECKPOINT_FILENAME, or LOCAL_CHECKPOINT.
LENGTH_PENALTY defaults to 0.0; change it here and rerun the runtime-config and evaluation cells to compare.
Other generation settings come from the qaware YAML; the source YAML is never edited.

This notebook only loads model weights and runs evaluation/benchmark. Training recipe metadata, optimizer restoration and per-rank RNG sidecars are not required. Architecture checks remain enabled; an actual architecture mismatch must be resolved with matching code/config.
Prepared data and DINO feature cache are reused and validated. No raw-image extraction is launched.
Results use the output directory already configured in the profile; reevaluation replaces metrics/predictions there.


In [ ]:
import os, json, sys
from pathlib import Path
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
GITHUB_REPO_URL = 'https://github.com/huyvanzzz/Group_HND_car.git'
GITHUB_BRANCH = 'huy-dinov2'
CHECKPOINT_HF_REPO_ID = ''  # Fill in your Hugging Face model repository.
CHECKPOINT_REPO_TYPE = 'model'
CHECKPOINT_FILENAME = 'finetune_latest.pt'  # Exact filename, including any repository subfolder.
LOCAL_CHECKPOINT = ''  # Optional existing .pt path, e.g. /kaggle/input/.../finetune_latest.pt.
LENGTH_PENALTY = 0.0
EVAL_BATCH_SIZE = 16
PREPARED_DATA_HF_REPO_ID = 'huyvanzzz/prepared_data'
CACHE_HF_REPO_ID = 'huyvanzzz/cache_dinov2'
REPO_DIR = Path('/kaggle/working/Efficient_VLM_For_Autonomous_Driving')
CONFIG_SOURCE = REPO_DIR / 'configs/dinov2_t5_efficient_mini_kaggle_2gpu_camera_query_qaware.yaml'
RUNTIME_CONFIG = Path('/kaggle/working/dinov2_camera_query_qaware_inference_runtime.yaml')
os.environ.update(GITHUB_REPO_URL=GITHUB_REPO_URL, GITHUB_BRANCH=GITHUB_BRANCH,
                  CONFIG=str(CONFIG_SOURCE), PYTHON_EXECUTABLE=sys.executable,
                  PYTHONPATH=str(REPO_DIR / 'src'), HF_HUB_DISABLE_XET='1', HF_HUB_DOWNLOAD_TIMEOUT='120')
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    print('HF_TOKEN secret unavailable; public repositories or an existing HF_TOKEN can still be used.')
print(json.dumps({'branch': GITHUB_BRANCH, 'length_penalty': LENGTH_PENALTY,
                  'checkpoint_filename': CHECKPOINT_FILENAME,
                  'hf_token_present': bool(os.environ.get('HF_TOKEN'))}, indent=2))


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving/.git ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git fetch origin "$GITHUB_BRANCH"
git checkout "$GITHUB_BRANCH"
git pull --ff-only origin "$GITHUB_BRANCH"
if ! "$PYTHON_EXECUTABLE" -m pip install --only-binary=:all: tokenizers==0.20.4; then
  echo "No compatible tokenizers==0.20.4 wheel. Select a Kaggle Python environment with a supported wheel; source builds are disabled." >&2
  exit 1
fi
"$PYTHON_EXECUTABLE" -m pip install --only-binary=tokenizers -e ".[mvpruner]" transformers==4.46.3 accelerate==1.15.0
"$PYTHON_EXECUTABLE" -m pip install -q pycocoevalcap
"$PYTHON_EXECUTABLE" - <<'PY'
import os, torch, tokenizers, transformers, accelerate
from efficient_vlm_ad.config import load_config
assert transformers.__version__ == "4.46.3", transformers.__version__
assert tokenizers.__version__ == "0.20.4", tokenizers.__version__
assert accelerate.__version__ == "1.15.0", accelerate.__version__
assert torch.cuda.is_available(), 'Enable Kaggle GPU'
assert torch.cuda.device_count() >= 1, 'Select at least one Kaggle GPU'
cfg = load_config(os.environ['CONFIG'])
assert cfg.runtime.precision == 'fp32'
assert cfg.model.vision.name == 'dinov2_small_avgpool'
assert cfg.model.text.model_id == 'google/t5-efficient-mini'
assert cfg.model.fusion.name == 'mvpruner_camera_query_qaware'
print('FP32', torch.__version__, transformers.__version__, tokenizers.__version__)
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
PY
nvidia-smi


In [ ]:
import json, math, os, shutil, subprocess, sys
from pathlib import Path
from huggingface_hub import hf_hub_download
REPO_SRC = '/kaggle/working/Efficient_VLM_For_Autonomous_Driving/src'
if REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
sys.path_importer_cache.pop(REPO_SRC, None)
from efficient_vlm_ad.config import load_config
from efficient_vlm_ad.cache import dinov2_manifest_fields, validate_manifest
from efficient_vlm_ad.modeling.vision import dinov2_processor_policy

def download_exact(repo_id, repo_type, destination, filenames):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        source = hf_hub_download(repo_id=repo_id, repo_type=repo_type, filename=filename)
        shutil.copy2(source, destination / filename)

def validate_prepared(prepared, full=False, expected_counts=None):
    prepared = Path(prepared)
    manifest = json.loads((prepared / 'manifest.json').read_text(encoding='utf-8'))
    if full and manifest.get('subset') != 'full':
        raise ValueError('Prepared data must be the full subset')
    counts = {}
    for split in ('train', 'val', 'test'):
        with (prepared / f'{split}.jsonl').open(encoding='utf-8') as stream:
            counts[split] = sum(1 for line in stream if line.strip() and json.loads(line))
        if counts[split] <= 0 or counts[split] != manifest['counts'][split]:
            raise ValueError(f'Prepared data count mismatch: {split}')
    if full and expected_counts and counts != expected_counts:
        raise ValueError(f'Prepared counts do not match expected dataset counts: {counts}')
    print('Prepared data counts:', counts)
    return manifest

def validate_cache(cfg):
    cache = Path(cfg.cache.dir)
    manifest = json.loads((cache / 'manifest.json').read_text(encoding='utf-8'))
    assert manifest['version'] == 2, 'DINO cache requires manifest v2'
    index = json.loads((cache / 'index.json').read_text(encoding='utf-8'))
    shape = manifest['shape']
    if len(shape) != 4 or shape[1:] != [6, 64, 384] or shape[0] != len(index) or shape[0] <= 0:
        raise ValueError(f'Expected DINO feature shape [N,6,64,384], got {shape}')
    if set(index.values()) != set(range(shape[0])):
        raise ValueError('Cache index must map every feature row exactly once')
    expected = dinov2_manifest_fields(cfg, shape, dinov2_processor_policy())
    validated = validate_manifest(cache / 'manifest.json', expected)
    if (cache / 'features.float16.memmap').stat().st_size != math.prod(shape) * 2:
        raise ValueError('Feature memmap byte length does not match manifest')
    prepared = Path(cfg.project.output_dir) / 'prepared_data'
    subset = json.loads((prepared / 'manifest.json').read_text(encoding='utf-8'))['subset']
    limit = None if subset == 'full' else int(cfg.training.max_steps or 64)
    for split in ('train', 'val', 'test'):
        with (prepared / f'{split}.jsonl').open(encoding='utf-8') as stream:
            for number, line in enumerate(line for line in stream if line.strip()):
                if limit is not None and number >= limit:
                    break
                record = json.loads(line)
                key = '|'.join(record['camera_paths'][camera] for camera in sorted(record['camera_paths']))
                if key not in index:
                    raise ValueError(f'Cache is missing {split} sample {number}')
    print('Validated DINO cache:', shape, validated.fingerprint)
    return validated

os.chdir(REPO_DIR)


In [ ]:
import json, math, os, yaml
from pathlib import Path
from efficient_vlm_ad.config import load_config
if isinstance(LENGTH_PENALTY, bool) or not isinstance(LENGTH_PENALTY, (int, float)) or not math.isfinite(LENGTH_PENALTY):
    raise ValueError('LENGTH_PENALTY must be a finite number')
if type(EVAL_BATCH_SIZE) is not int or EVAL_BATCH_SIZE < 1:
    raise ValueError('EVAL_BATCH_SIZE must be a positive integer')
raw = yaml.safe_load(CONFIG_SOURCE.read_text(encoding='utf-8'))
raw['generation']['length_penalty'] = float(LENGTH_PENALTY)
RUNTIME_CONFIG.write_text(yaml.safe_dump(raw, sort_keys=False), encoding='utf-8')
cfg = load_config(RUNTIME_CONFIG)
assert cfg.model.fusion.name == 'mvpruner_camera_query_qaware'
os.environ['CONFIG'] = str(RUNTIME_CONFIG)
os.environ['PROFILE'] = cfg.project.output_dir
print(json.dumps({'config': str(RUNTIME_CONFIG), 'output_dir': cfg.project.output_dir,
                  'generation': raw['generation'], 'eval_batch_size': EVAL_BATCH_SIZE}, indent=2))


In [ ]:
from efficient_vlm_ad.checkpointing import read_checkpoint_metadata, validate_checkpoint_signature
if LOCAL_CHECKPOINT:
    checkpoint_path = Path(LOCAL_CHECKPOINT).expanduser().resolve()
else:
    if not CHECKPOINT_HF_REPO_ID:
        raise ValueError('Set CHECKPOINT_HF_REPO_ID or LOCAL_CHECKPOINT in the settings cell')
    checkpoint_path = Path(hf_hub_download(repo_id=CHECKPOINT_HF_REPO_ID,
                                          repo_type=CHECKPOINT_REPO_TYPE,
                                          filename=CHECKPOINT_FILENAME))
if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)
metadata = read_checkpoint_metadata(checkpoint_path, map_location='cpu')
validate_checkpoint_signature(metadata, cfg=cfg)
if metadata.get('stage') != 'finetune':
    raise ValueError('Select a finetune checkpoint for this inference notebook')
print(json.dumps({'checkpoint': str(checkpoint_path), 'stage': metadata.get('stage'),
                  'architecture_matches': True}, indent=2))


In [ ]:
prepared = Path(cfg.project.output_dir) / 'prepared_data'
cache = Path(cfg.cache.dir)
prepared_files = ('manifest.json', 'train.jsonl', 'val.jsonl', 'test.jsonl')
cache_files = ('features.float16.memmap', 'index.json', 'manifest.json')
for repo, destination, filenames in ((PREPARED_DATA_HF_REPO_ID, prepared, prepared_files),
                                     (CACHE_HF_REPO_ID, cache, cache_files)):
    missing = [name for name in filenames if not (destination / name).is_file()]
    if missing:
        if not repo:
            raise FileNotFoundError(f'Missing files in {destination}: {missing}; configure the dataset repo')
        download_exact(repo, 'dataset', destination, missing)
validate_prepared(prepared, full=True, expected_counts=cfg.data.expected_counts)
validate_cache(cfg)


In [ ]:
import subprocess, sys
commands = [
    [sys.executable, '-m', 'efficient_vlm_ad', 'evaluate', '--config', str(RUNTIME_CONFIG),
     '--checkpoint', str(checkpoint_path), '--batch-size', str(EVAL_BATCH_SIZE)],
    [sys.executable, '-m', 'efficient_vlm_ad', 'benchmark', '--config', str(RUNTIME_CONFIG),
     '--checkpoint', str(checkpoint_path)],
]
for command in commands:
    subprocess.run(command, cwd=REPO_DIR, check=True)
output_dir = Path(cfg.project.output_dir)
for filename in ('metrics_display.json', 'metrics.json', 'benchmark.json'):
    path = output_dir / filename
    if path.is_file():
        print(filename)
        print(path.read_text(encoding='utf-8'))
print('Predictions:', output_dir / 'predictions.jsonl')
